# Sentinel 1 & 2: Fusion

In [ ]:
import os
import sys
from pathlib import Path
import importlib
import torch

## Setup

In [ ]:
root_path = "/content/drive/MyDrive/MSc/Flood-Mapping"  #@param {type:"string", multiline:true}
mount_drive = True  #@param {type:"boolean"}
clone_repo = False  #@param {type:"boolean"}
download_results = False  #@param {type:"boolean"}
run_training = False  #@param {type:"boolean"}

import sys
from pathlib import Path

REPO_URL = "https://github.com/TAX2310/Flood-Mapping.git"

if not mount_drive and not clone_repo:
    raise ValueError("Either mount_drive or clone_repo must be True.")

if mount_drive:
    from google.colab import drive
    drive.mount("/content/drive")
else:
    root_path = "Flood-Mapping"

repo = Path(root_path)

if clone_repo and not repo.exists():
    !git clone $REPO_URL $root_path

assert repo.exists(), f"Repo not found at {repo}. Enable clone_repo or fix root_path."

sys.path.append(str(repo))
from src.config import Fusion_CFG

cfg = Fusion_CFG()
cfg.ROOT = repo
cfg.DEVICE = "cuda" if torch.cuda.is_available() else "cpu"


## Install Dependencies

In [ ]:
requirements = cfg.ROOT / "requirements.txt"
!pip install -r {requirements}

## Imports

In [ ]:
import src.data.sturm_fusion as SturmFusion
import src.train.training as training
import src.test.testing as testing
import src.util.io as io
import src.util.plotting as plot

## Load Dataset

In [ ]:
data_root = SturmFusion.download_and_extract_dataset(cfg)

img_dir = cfg.S1_PATH
mask_dir = cfg.MASK_PATH

print("Image dir exists:", img_dir.exists())
print("Mask dir exists:", mask_dir.exists())

print("Num images:", len(list(img_dir.glob("*.tif"))))
print("Num masks:", len(list(mask_dir.glob("*.tif"))))

if download_results:
    SturmFusion.download_and_extract_results(cfg)

## Train Model

In [ ]:
if run_training:
    for learning_rate in cfg.LEARNING_RATES:
        for batch_size in cfg.BATCH_SIZES:
            for weight_decay in cfg.WEIGHT_DECAYS:
                for dropout_rate in cfg.DROPOUT_RATES:
                    cfg.LR = learning_rate
                    cfg.BATCH_SIZE = batch_size
                    cfg.WEIGHT_DECAY = weight_decay
                    cfg.DROPOUT_RATE = dropout_rate
                    training.train_from_file(cfg)

## Training Results

In [ ]:
plot.plot_hp_comparison_bar(cfg, save_path=cfg.FIG_EXPORTS_DIR/"fusion_hp_iou_f1.pdf")

In [ ]:
plot.view_training_metrics(cfg)

## Test Model

In [ ]:
testing.test_model(cfg, cfg.FUSION_MODEL)

In [ ]:
testing.select_model_to_test(cfg)

## Run Inference

In [ ]:
import src.inference.inference as inference

#samples = ["EMSR570_AOI02_07_03_2_1.tif", "EMSR470_AOI01_46_07_2_1.tif", "EMSR470_AOI01_47_10_1_2.tif"]

samples = ["EMSR470_AOI01_46_07_2_1.tif", "EMSR441_AOI05_2_3_2_2.tif", "EMSR570_AOI02_07_03_2_1.tif"]

samples = ["EMSR470_AOI01_29_13_2_2.tif", "EMSR407_AOI01_03_17_2_1.tif", "EMSR470_AOI01_10_25_1_1.tif", "EMSR470_AOI01_47_10_1_2.tif", "EMSR629_AOI01_09_01_1_2.tif"]

results = inference.inference(cfg, cfg.FUSION_MODEL, samples)
all_results = inference.inference(cfg, cfg.FUSION_MODEL)
io.create_inference_results_csv(cfg, all_results, cfg.METADATA_CSV, cfg.FUSION_TEST_RESULTS_CSV)

In [ ]:
plot.plot_sample_results(cfg, results)

## Fusion Model Evaluation

In [ ]:
plot.plot_metric_distribution_from_csv(cfg.FUSION_TEST_RESULTS_CSV, save_path=cfg.FIG_EXPORTS_DIR/"fusion_iou_dist.pdf")

In [ ]:
plot.plot_iou_vs_flood_scatter([cfg.FUSION_TEST_RESULTS_CSV], save_path=cfg.FIG_EXPORTS_DIR/"fusion_scatter.pdf", show_legend=False)

In [ ]:
plot.plot_iou_vs_flood_median([cfg.FUSION_TEST_RESULTS_CSV],bin_width=10, show_scatter=True, band_alpha=0.2, scatter_alpha=0.25, save_path=cfg.FIG_EXPORTS_DIR/"median_fusion.pdf", show_legend=True)

In [ ]:
plot.plot_average_iou_per_event(cfg.FUSION_TEST_RESULTS_CSV, save_path=cfg.FIG_EXPORTS_DIR/"fusion_iou_per_event.pdf")

## Compare S1 / S2 / Fusion

In [ ]:
plot.plot_iou_vs_flood_scatter([cfg.S2_TEST_RESULTS_CSV,
                                cfg.S1_TEST_RESULTS_CSV,
                                cfg.FUSION_TEST_RESULTS_CSV], figsize=(16, 12), save_path=cfg.FIG_EXPORTS_DIR/"scatter_iou_flood_all.pdf", show_legend=True)

In [ ]:
plot.plot_iou_vs_flood_median([cfg.S2_TEST_RESULTS_CSV,
                                cfg.S1_TEST_RESULTS_CSV,
                                cfg.FUSION_TEST_RESULTS_CSV],bin_width=10, show_scatter=False, band_alpha=0.0, save_path=cfg.FIG_EXPORTS_DIR/"median_iou_flood_all.pdf", show_legend=True)

In [ ]:
plot.plot_fusion_improvement_distribution([cfg.S2_TEST_RESULTS_CSV,
                                cfg.S1_TEST_RESULTS_CSV,
                                cfg.FUSION_TEST_RESULTS_CSV], save_path=cfg.FIG_EXPORTS_DIR/"fusion_improvement_distribution.pdf")

## Sample Prediction Visualization

In [ ]:
import src.inference.inference as inference

samples = ["EMSR570_AOI02_07_03_2_1.tif", "EMSR470_AOI01_46_07_2_1.tif", "EMSR470_AOI01_47_10_1_2.tif"]

results = inference.inference(cfg, cfg.FUSION_MODEL, samples)

plot.plot_sample_results(cfg, results)